In [27]:
# 데이터 준비 (파일에서 배치를 적용한 읽기 도구 만들기)

from tensorflow import keras as tf_keras

train_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/train', batch_size=32
)
validation_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/val', batch_size=32
)
test_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/test', batch_size=32
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [28]:
# BoW 모델 기반 텍스트 데이터 인코딩 도구 학습

max_length = 600 # 한 문장의 길이 (토큰 갯수)
max_tokens = 20000 # 단어 사전의 크기 (토큰 갯수)
text_vectorization = tf_keras.layers.TextVectorization(
    max_tokens=max_tokens, # 단어 사전에 포함될 단어 갯수 (빈도수 높은 순)
    output_mode='int', # 각 단어의 단어 사전에 지정된 번호 인코딩
    output_sequence_length=max_length # 한 문장을 구성하는 단어 갯수
)

only_text_dataset = train_dataset.map(lambda x, y: x)
text_vectorization.adapt( only_text_dataset ) # 학습을 통해 단어 사전 구성

In [29]:
# 데이터 셋의 각 데이터에 대해 인코딩 처리

encoded_train_dataset = train_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )
encoded_validation_dataset = validation_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )
encoded_test_dataset = test_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )

In [ ]:
# 인코딩 결과 확인 : 각 문장은 단어 사전에서 그 단어에 해당하는 번호로 기록
for features, labels in encoded_train_dataset:
    print(features[0])
    break

tf.Tensor(
[   10    42  1592    11     1   746     5    50   821  9462  1514    49
     4  2384     5  8065 11834    11    18     7   122   253   119     2
  5543     5     2   173    38    10    69    76    55   737  2480   142
   388   285 14841     1    36     2   240  4765   321    18   120   598
    29  1551     6    27  1272    10    89   103    48    24    98    50
  4765   321    95   594    19    11    14    63    83    13    13    10
   477    76    79    98  1621  7288   220     4  3091   821    36    81
     1  8419    12  1493    99    32    45     2   173  2619     6     1
    54     2     1     1    43  1621  4952   187    35  2779    91     2
    18   606     1    10   474     2   757   139    27   332     2  1105
    16    11   425     5     4    20    28    41    56   514     1     2
   115    16    11    18    36    47  3645  1548    22 13615    17     2
     1     5   618   122  1775    19    70   578    49     2   214     5
    11    18    14    14    48     4   7

In [ ]:
# glove.11.9b.100d or glove.6b.100d.txt 파일 준비
# 다운로드 링크 : https://nlp.stanford.edu/projects/glove/

In [22]:
# 사전 학습 임베딩 준비

import numpy as np

embedding_index = {} # { 단어(토큰) : 임베딩 벡터 } 형식의 딕셔너리 준비
with open('data-files/glove.11.9b.100d.txt') as f:
    # for idx, line in enumerate(f):
    for idx, line in enumerate(f):
        token_and_embeddings = line.split(' ')
        word, weights = token_and_embeddings[0], token_and_embeddings[1:]
        
        weights = np.array([float(weight) for weight in weights])

        embedding_index[word] = weights
        

In [ ]:
# 사전 학습 임베딩 확인
print( f"단어 갯수 : {len( embedding_index.keys() )}" )
print( embedding_index[ list(embedding_index.keys())[0] ] )

단어 갯수 : 1291147
[ 0.306717 -0.32053  -0.393647  0.082826  0.073522 -0.409154 -0.265564
 -0.23694  -0.305832  0.74529   0.214341  0.276781 -0.152797 -0.127524
  0.119525  0.640965 -0.175869  0.160711  0.477978 -0.160939 -0.150093
  0.674601 -0.099565  0.021882 -0.032771  0.368641 -0.087019 -0.133326
  0.170143  0.156934  0.677506 -0.099686  0.392113  0.373434 -5.736062
  0.413845  0.477368 -0.041697  0.383109  0.120152 -0.20947   0.605104
  0.236353  0.151131 -0.508865  0.671239 -0.300263 -0.267927  2.549487
  0.067177  0.217224 -0.031316  0.05231   0.119321 -0.332154 -0.807904
 -0.546453 -0.044392 -0.281657  0.286647  0.325775 -0.02196  -0.636903
 -0.268063  0.247956 -0.402493  0.276707 -0.275139  0.201159  0.082844
  0.591695 -0.017127 -0.092269  0.392008  0.078245 -0.049907  0.235151
  0.457376 -0.111987 -0.05691   0.065092  0.106512  0.98337   0.608167
 -0.250386 -0.44968   0.185177  0.056664  0.296632 -0.244439 -0.019507
  0.324114  0.460377 -0.041331 -0.333933  0.062373 -0.114783 